## Example 1: Basic zarr array creation

In [ ]:
import zarr
import numpy as np

# Example 1: Basic zarr array creation
print("Basic zarr array creation:")

# Create a simple zarr array with explicit store
store1 = {}
z = zarr.zeros((1000, 1000), store=store1, chunks=(100, 100), dtype='f4')
print(f"Created zarr array: shape={z.shape}, dtype={z.dtype}")
print(f"Chunks: {z.chunks}")

# Use compressors instead of compressor for Zarr v3
if hasattr(z, 'compressors'):
    print(f"Compressors: {z.compressors}")
else:
    print(f"Compressor: {z.compressor}")

# Create array with data
data = np.random.random((50, 50)).astype('f4')
store2 = {}
z_data = zarr.array(data, store=store2, chunks=(25, 25))
print(f"\nArray from data: shape={z_data.shape}")
print(f"Chunks: {z_data.chunks}")
print(f"Data sample:\n{z_data[0:3, 0:3]}")

# Basic array operations
z[0:100, 0:100] = 1.5
print(f"\nAssigned values to slice [0:100, 0:100]")
print(f"Sample values:\n{z[0:5, 0:5]}")

print("\nBasic array creation completed")

## Example 2: Compression and storage

In [ ]:
import zarr
import numpy as np

# Example 2: Compression and storage
print("Compression and storage:")

data = np.random.random((100, 100)).astype('f8')

# Create arrays demonstrating different storage options
print("Default array with compression:")
z_default = zarr.array(data, chunks=(50, 50))
print(f"  Shape: {z_default.shape}")
print(f"  Chunks: {z_default.chunks}")
print(f"  Data type: {z_default.dtype}")

# Memory storage vs directory storage
print("\nStorage backends:")

# In-memory storage using dictionary
memory_store = {}
z_memory = zarr.zeros((100, 100), store=memory_store, chunks=(50, 50))
z_memory[0:10, 0:10] = 42
print(f"Memory store type: {type(memory_store)}")
print(f"Sample data:\n{z_memory[0:3, 0:3]}")

# Create another array with different chunk size
print("\nArray with different chunking:")
z_large_chunks = zarr.array(data, chunks=(100, 100))
print(f"  Shape: {z_large_chunks.shape}")
print(f"  Chunks: {z_large_chunks.chunks}")
print(f"  Number of chunks: {z_large_chunks.nchunks}")

print("\nCompression and storage completed")

## Example 3: Groups and hierarchical data

In [ ]:
import zarr
import numpy as np

# Example 3: Groups and hierarchical data
print("Groups and hierarchical data:")

# Create a group
root = zarr.group()

# Create subgroups and arrays
root.create_group('experiment_1')
root.create_group('experiment_2')

# Add arrays to groups
root['experiment_1'].create_array('data', data=np.random.random((50, 50)), chunks=(25, 25))
root['experiment_1'].create_array('labels', data=np.arange(50), chunks=(25,))

root['experiment_2'].create_array('data', data=np.random.random((30, 40)), chunks=(15, 20))

# Explore the hierarchy
print(f"Root group: {root}")
print(f"Groups: {list(root.groups())}")
print(f"Arrays in experiment_1: {list(root['experiment_1'].arrays())}")

# Access nested data
exp1_data = root['experiment_1']['data']
print(f"Experiment 1 data shape: {exp1_data.shape}")
print(f"Sample data: {exp1_data[0:3, 0:3]}")

# Group attributes
root.attrs['description'] = 'Scientific experiment data'
root['experiment_1'].attrs['date'] = '2024-01-01'
print(f"Root attributes: {dict(root.attrs)}")

print("Hierarchical data completed")

## Example 4: Array slicing and indexing

In [ ]:
import zarr
import numpy as np

# Example 4: Array slicing and indexing
print("Array slicing and indexing:")

# Create a 3D array
z3d = zarr.zeros((100, 200, 50), chunks=(50, 100, 25), dtype='i4')

# Fill with sample data
for i in range(100):
    z3d[i, :, :] = i

print(f"3D array shape: {z3d.shape}")
print(f"3D array chunks: {z3d.chunks}")

# Various slicing operations
print("\nSlicing operations:")
print(f"z3d[0, :5, :3]:\n{z3d[0, :5, :3]}")
print(f"z3d[10:15, 0, 0]: {z3d[10:15, 0, 0]}")

# Boolean indexing
mask = z3d[:, 0, 0] > 50
selected = z3d[mask, 0, 0]
print(f"Boolean indexing (values > 50): {selected[:5]}")

# Fancy indexing
indices = [10, 20, 30, 40]
fancy_result = z3d[indices, 0, 0]
print(f"Fancy indexing: {fancy_result}")

# Partial assignment
z3d[90:100, 0:10, 0:5] = 999
print(f"After assignment: {z3d[95, 5, 2]}")

# Array statistics
print(f"\nArray statistics:")
print(f"Max value: {np.max(z3d[:])}")
print(f"Min value: {np.min(z3d[:])}")

print("Slicing and indexing completed")

## Example 5: Performance and chunking strategies

In [ ]:
import zarr
import numpy as np
import time

# Example 5: Performance and chunking strategies
print("Performance and chunking strategies:")

# Test different chunking strategies
data = np.random.random((1000, 1000)).astype('f4')

chunk_strategies = [
    (100, 100),   # Square chunks
    (1000, 100),  # Row chunks
    (100, 1000),  # Column chunks
    (200, 200),   # Larger squares
]

print("Chunking strategy performance:")
for i, chunks in enumerate(chunk_strategies):
    start_time = time.time()
    z = zarr.array(data, chunks=chunks)
    
    # Simulate column-wise access
    col_sum = np.sum(z[:, 0:100])
    
    end_time = time.time()
    
    print(f"Strategy {i+1} {chunks}: {end_time - start_time:.4f}s")
    print(f"  Chunk shape: {z.chunks}")
    print(f"  Number of chunks: {z.nchunks}")
    print()

# Memory usage comparison
print("Memory usage patterns:")
sparse_array = zarr.zeros((10000, 10000), chunks=(1000, 1000))
sparse_array[0:100, 0:100] = 1.0
sparse_array[5000:5100, 5000:5100] = 2.0

print(f"Sparse array shape: {sparse_array.shape}")
print(f"Actual memory usage is minimal due to chunking")
print(f"Non-zero chunks only store data")

# Array comparison with different chunk sizes
print(f"\nChunk size comparison:")
small_chunks = zarr.zeros((500, 500), chunks=(50, 50))
large_chunks = zarr.zeros((500, 500), chunks=(250, 250))

test_pattern = np.tile(np.arange(100), (500, 5))
small_chunks[:] = test_pattern
large_chunks[:] = test_pattern

print(f"Small chunks: shape={small_chunks.shape}, chunks={small_chunks.chunks}, nchunks={small_chunks.nchunks}")
print(f"Large chunks: shape={large_chunks.shape}, chunks={large_chunks.chunks}, nchunks={large_chunks.nchunks}")

print("\nPerformance analysis completed")